# Source officielle Human-Centered-xAI

> Clonage, résolution de chemins, import dynamique et construction du modèle AAE depuis la branche `Arda`.


In [ ]:
#| default_exp source


In [ ]:
#| export
"""Resolve, clone, import, and instantiate the upstream Human-Centered-xAI AAE."""


import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from types import ModuleType

from torch import nn

from tell_me_why.config import HumanCenteredAAEConfig

HUMAN_CENTERED_XAI_REPO_URL = "https://github.com/LucaLaFisca/Human-Centered-xAI.git"
HUMAN_CENTERED_XAI_BRANCH = "Arda"
HUMAN_CENTERED_XAI_MODEL_FILE = "modelAAE_DROPOUT.py"
PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
DEFAULT_HUMAN_CENTERED_XAI_REPO = PROJECT_ROOT.parent / "Human-Centered-xAI"
DEFAULT_HUMAN_CENTERED_XAI_MODEL = (
    DEFAULT_HUMAN_CENTERED_XAI_REPO / HUMAN_CENTERED_XAI_MODEL_FILE
)


def ensure_human_centered_xai_repo(
    target_dir: str | Path | None = None,
    *,
    repo_url: str = HUMAN_CENTERED_XAI_REPO_URL,
    branch: str = HUMAN_CENTERED_XAI_BRANCH,
    update: bool = False,
) -> Path:
    """Clone or optionally update the official Human-Centered-xAI branch."""

    path = Path(target_dir).expanduser().resolve() if target_dir else DEFAULT_HUMAN_CENTERED_XAI_REPO
    if path.exists():
        if update:
            subprocess.run(["git", "fetch", "origin", branch], cwd=path, check=True)
            subprocess.run(["git", "checkout", branch], cwd=path, check=True)
            subprocess.run(["git", "pull", "--ff-only", "origin", branch], cwd=path, check=True)
        return path

    subprocess.run(
        ["git", "clone", "--branch", branch, "--single-branch", repo_url, str(path)],
        check=True,
    )
    return path


def resolve_human_centered_aae_path(model_path: str | Path | None = None) -> Path:
    """Resolve the absolute path to `modelAAE_DROPOUT.py`."""

    if model_path is None:
        model_path = os.getenv("HUMAN_CENTERED_XAI_MODEL_PATH")
    if model_path is not None:
        path = Path(model_path)
    else:
        repo_dir = os.getenv("HUMAN_CENTERED_XAI_REPO_DIR")
        path = (
            Path(repo_dir).expanduser() / HUMAN_CENTERED_XAI_MODEL_FILE
            if repo_dir
            else DEFAULT_HUMAN_CENTERED_XAI_MODEL
        )
    path = path.expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            "Cannot find modelAAE_DROPOUT.py. Pass `model_path` or set "
            "`HUMAN_CENTERED_XAI_MODEL_PATH`. Official source: "
            f"{HUMAN_CENTERED_XAI_REPO_URL} branch `{HUMAN_CENTERED_XAI_BRANCH}`. "
            "You can clone it with `ensure_human_centered_xai_repo()`."
        )
    return path


def load_human_centered_aae_module(model_path: str | Path | None = None) -> ModuleType:
    """Import `Human-Centered-xAI/modelAAE_DROPOUT.py` as a Python module."""

    path = resolve_human_centered_aae_path(model_path)
    module_name = f"tell_me_why_human_centered_aae_{abs(hash(path))}"
    if module_name in sys.modules:
        return sys.modules[module_name]

    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create an import spec for {path}.")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    try:
        spec.loader.exec_module(module)
    except ModuleNotFoundError as error:
        missing = error.name or str(error)
        raise ModuleNotFoundError(
            f"`{path}` requires `{missing}`. Install the fastai environment "
            "declared by this package before loading the AAE."
        ) from error
    return module


def build_human_centered_aae(
    config: HumanCenteredAAEConfig | None = None,
    *,
    model_path: str | Path | None = None,
) -> nn.Module:
    """Instantiate the `AAE` class from `modelAAE_DROPOUT.py`."""

    module = load_human_centered_aae_module(model_path)
    if not hasattr(module, "AAE"):
        raise AttributeError(f"{resolve_human_centered_aae_path(model_path)} has no AAE class.")
    return module.AAE(**(config or HumanCenteredAAEConfig()).to_kwargs())
